# 第 10 章 · 评估与测试工程

> **你已会**：用 Runner 执行 Agent、读工具调用事件、用回调阻止操作。
>
> **本章后你能**：写三条可回放案例，真正运行 `adk eval`，从失败报告定位到工具或答案，再把业务状态断言加入发布检查。
>
> **边界**：离线模型替身验证的是运行和评估链路；真实模型评估只覆盖选定的数据。分数高不等于事实正确，更不等于可安全上线。

## 1. 问题：它说“已退款”，你相信什么？

用户说：“请为订单 A1001 创建 299 元退款申请。”回答是“退款申请已提交”。有三个独立的问题：

1. 它有没有调用正确的工具，参数是什么？
2. 回答与我们期望的表达有多接近？
3. 退款账本是否真的新增了正确记录？

轨迹、文本、业务状态分别提供证据。先从只读订单查询学会前两种，再进入有副作用的退款案例，避免一上来同时调试模型、工具和评估脚本。

## 2. 预测：只写三条，分别检查三种决策

| 用户输入 | 期望工具 | 期望回答 | 为什么需要 |
|---|---|---|---|
| 查询订单 A1001 | query_order(A1001) | 已签收 | 正常路径 |
| 查询订单 Z9999 | query_order(Z9999) | 未找到订单 | 工具返回业务错误 |
| 帮我查一下订单 | 不调用 | 请提供订单号 | 信息不足时先澄清 |

先预测：如果 Agent 背出了正确答案，却跳过 `query_order`，哪项指标应该失败？

### EvalSet 的最小结构

| 字段 | 含义 |
|---|---|
| `eval_set_id` | 数据集标识 |
| `eval_cases[]` | 一个案例；`eval_id` 是案例 ID |
| `conversation[]` | 多轮交互列表；每轮用 `user_content` 表示输入 |
| `intermediate_data.tool_uses` | 期望的工具调用轨迹（工具名、参数、顺序） |
| `final_response` | 参考回答 |
| `session_input` | 会话初始状态（如用户、租户） |
| `test_spec` | 本项目附加的业务断言；官方 CLI 不执行 |

新增案例时，先复制一个最接近的案例，再改 `eval_id`、`user_content`、`tool_uses`、`final_response`；`test_spec` 只放本项目自己的业务检查。

In [1]:
import json
import sys
from pathlib import Path

root = Path.cwd()
if not (root / "official_eval_example").exists():
    root = root / "google-adk"
assert (root / "official_eval_example").is_dir(), "请从仓库或 google-adk 目录运行"
sys.path.insert(0, str(root))
small = json.loads((root / "official_eval_example/three_cases.evalset.json").read_text(encoding="utf-8"))
for case in small["eval_cases"]:
    turn = case["conversation"][0]
    print(case["eval_id"], turn["user_content"]["parts"][0]["text"],
          turn["intermediate_data"]["tool_uses"])

order_found 查询订单 A1001 [{'name': 'query_order', 'args': {'order_id': 'A1001'}}]
order_missing 查询订单 Z9999 [{'name': 'query_order', 'args': {'order_id': 'Z9999'}}]
need_order_id 帮我查一下订单 []


## 3. 实验：真正运行官方 CLI，模型费用为零

[`official_eval_example/agent.py`](official_eval_example/agent.py) 使用 `OrderModel(BaseLlm)` 产生确定性的模型响应。**Runner、FunctionTool、Session、事件和评估器仍是实际 ADK**。它用于教学和基础设施回归，不测 DeepSeek 的能力。

ADK 2.6.3 的默认指标注册表导入 Vertex SDK；完整 `[eval]` extra 又可能与课程的 LiteLLM 版本冲突。因此把 CLI 依赖放入专用 Docker 镜像。安装时需要网络，运行时使用 `--network none`，无须任何密钥，也不修改主环境。

在仓库根目录的 PowerShell 执行：

```powershell
docker build -f google-adk/official_eval_example/Dockerfile -t adk-course-eval:2.6.3 .
docker run --rm --network none --name adk-course-eval-green adk-course-eval:2.6.3
```

镜像的入口就是 `adk eval official_eval_example`。完整原生命令是：

```text
adk eval official_eval_example official_eval_example/three_cases.evalset.json --config_file_path official_eval_example/evaluation_config.json --print_detailed_results
```

**预期证据**：三条 `PASSED`；工具轨迹和回答相似度均为 1。环境、复现实验和已验证范围见 [CLI 实验说明](official_eval_example/README.md)。

## 4. 反例：回答满分，轨迹零分

不要先改阈值。先注入一个明确的错误：跳过工具，直接回答。

```powershell
docker run --rm --network none --name adk-course-eval-broken -e ADK_EVAL_INJECT_BUG=skip_tool adk-course-eval:2.6.3 official_eval_example/three_cases.evalset.json:order_found --config_file_path official_eval_example/evaluation_config.json --print_detailed_results
```

`:order_found` 只运行一条；多个 ID 用逗号连接。预期 `response_match_score=1`，`tool_trajectory_avg_score=0`，案例 `FAILED`。这就是“答对了，但执行过程不符合要求”。

修复实验是取消故障变量，然后回归全部三条：

```powershell
docker run --rm --network none --name adk-course-eval-repaired adk-course-eval:2.6.3
```

**容易误判的证据**：本例使用的 ADK 2.6.3 CLI 即使报告 `FAILED`，进程也可能返回 0。`0` 只说明命令完成；CI 应解析本次生成的报告。下面的脚本会执行通过→失败→修复→单条选择，保存四份原始官方报告，并检查每次结果是否符合预测：

```powershell
.venv\Scripts\python.exe google-adk/official_eval_example/verify_cli.py --backend docker
```

`check_result.py` 把单份官方报告转换成明确的 CI 退出码：全通过为 0，失败/未评估为 1，报告损坏或无案例为 2。

In [2]:
# 运行上一节 verify_cli.py 后，再读本次生成的证据。
report_path = root / "official_eval_example/reports/verification.json"
if report_path.exists():
    verification = json.loads(report_path.read_text(encoding="utf-8"))
    for run in verification["runs"]:
        print(run["run"], "CLI exit:", run["cli_exit_code"],
              "cases passed:", run["passed"], run["cases"])
else:
    print("尚未运行官方 CLI；请先执行上一节命令。此处不会把缺报告算成通过。")

green CLI exit: 0 cases passed: True [{'eval_id': 'order_found', 'status': 1, 'metrics': {'tool_trajectory_avg_score': 1.0, 'response_match_score': 1.0}}, {'eval_id': 'order_missing', 'status': 1, 'metrics': {'tool_trajectory_avg_score': 1.0, 'response_match_score': 1.0}}, {'eval_id': 'need_order_id', 'status': 1, 'metrics': {'tool_trajectory_avg_score': 1.0, 'response_match_score': 1.0}}]
broken CLI exit: 0 cases passed: False [{'eval_id': 'order_found', 'status': 2, 'metrics': {'tool_trajectory_avg_score': 0.0, 'response_match_score': 1.0}}]
repaired CLI exit: 0 cases passed: True [{'eval_id': 'order_missing', 'status': 1, 'metrics': {'tool_trajectory_avg_score': 1.0, 'response_match_score': 1.0}}, {'eval_id': 'need_order_id', 'status': 1, 'metrics': {'tool_trajectory_avg_score': 1.0, 'response_match_score': 1.0}}, {'eval_id': 'order_found', 'status': 1, 'metrics': {'tool_trajectory_avg_score': 1.0, 'response_match_score': 1.0}}]
selected CLI exit: 0 cases passed: True [{'eval_id': '

### 4.1 怎么从报告找到原因

| 看到的证据 | 下一步检查 | 不应得出的结论 |
|---|---|---|
| 轨迹 0、回答 1 | 实际与期望工具名、顺序、参数 | 回答正确，所以可以忽略轨迹 |
| 轨迹 1、回答低分 | 先读真实回答与参考答案 | 低分一定代表业务失败 |
| 两项都是 1 | 继续查业务副作用与权限 | 系统对所有输入都正确 |
| NOT_EVALUATED | 查看错误与缺少的输入 | 等于零分，或者等于通过 |

`response_match_score` 是 Unicode-aware ROUGE-1 的词元重叠指标。它不检查事实、授权、数据库结果，也不理解“已退款”和“尚未退款”的业务差异。阈值应结合人工审查后的案例校准，不应用调低阈值掩盖执行错误。

## 5. 迁移：从三条到售后的 26 条

现在再打开 [`after_sales.evalset.json`](eval_example/eval_sets/after_sales.evalset.json)。先选一条订单案例，再加缺参数、工具失败、退款、攻击、多轮指代。新增案例的理由应来自**新的失败方式**，而不是为了凑数量。

原生 EvalSet 的 `user_content` 是输入，`intermediate_data.tool_uses` 是期望轨迹，`final_response` 是参考回答。此项目另外使用 `test_spec` 保存业务断言，官方 CLI 不执行这些自定义断言。

26 条、28 轮是本仓库的教学覆盖集；`dataset_min_cases=20` 只是完整流程的约定，不能据此声称统计充分或达到生产标准。

In [3]:
from collections import Counter
from eval_example.run_quality_eval import validate_dataset

large = json.loads((root / "eval_example/eval_sets/after_sales.evalset.json").read_text(encoding="utf-8"))
errors = validate_dataset(large)
assert not errors, errors
print("案例数：", len(large["eval_cases"]))
print("类别：", dict(Counter(c["test_spec"]["category"] for c in large["eval_cases"])))
refund = next(c for c in large["eval_cases"] if c["eval_id"] == "refund_approved")
print("合法退款必须新增：", refund["test_spec"]["turns"][0]["expected_refunds_created"])

案例数： 26
类别： {'normal': 8, 'edge': 6, 'tool_failure': 2, 'high_risk': 3, 'safety': 3, 'safety_negative': 1, 'multi_turn': 2, 'multi_tool': 1}
合法退款必须新增： [{'idempotency_key': 'refund-a1001-001', 'order_id': 'A1001', 'amount': 299, 'reason': '按键失灵', 'status': '待审核'}]


## 6. 实验：把“任务成功”连接到业务状态

先预测下面三个结果：①回答说成功但账本为空；②应该拒绝却新增退款；③指定订单、金额、原因都正确地新增。只有第三种应通过。

`check_refund_effects` 对比执行前后的账本快照：只允许声明的新增，禁止遗漏、额外写入或覆盖旧记录。每条案例独立清空账本，同一案例的多轮继续共享状态。

In [4]:
from eval_example.business_checks import check_refund_effects
from eval_example.tools import create_refund_request, get_refund_requests, reset_demo_state

expected = [{"idempotency_key": "lesson-10", "order_id": "A1001", "amount": 299,
             "reason": "按键失灵", "status": "待审核"}]
reset_demo_state()
print("声称成功，但没有写：", check_refund_effects({}, get_refund_requests(), expected))
create_refund_request("A1001", 299, "按键失灵", "APPROVED", "lesson-10")
print("应该拒绝，却发生写：", check_refund_effects({}, get_refund_requests(), []))
print("按预期新增：", check_refund_effects({}, get_refund_requests(), expected))
reset_demo_state()

声称成功，但没有写： False
应该拒绝，却发生写： False
按预期新增： True


`APPROVED` 是演示审批码，不是真实授权。工具本体会重查资格、金额、演示审批码，并拒绝同一幂等键对应不同请求；生产服务仍需要可信身份、授权、事务和持久化。

运行离线回归，确认这两个反例、工具绕过回调的拒绝、幂等冲突、报告口径和 ROUGE 兼容性：

```powershell
Set-Location google-adk
..\.venv\Scripts\python.exe -m unittest discover -s eval_example/tests -v
..\.venv\Scripts\python.exe -m eval_example.run_quality_eval --dry-run
```

本项目复用的 ROUGE 私有 API 集中在 `rouge_compat.py`，只声明验证过 ADK 2.6.3；升级时先跑契约测试，不能假设私有方法永远兼容。

## 7. 真实模型：先冒烟，再决定是否全量

离线替身通过后，才值得支付模型费用。可选路线 A：在专用 CLI 环境设置 `ADK_EVAL_BACKEND=deepseek`、传入 `DEEPSEEK_API_KEY`，先只跑 `:order_found`。详细命令见 [CLI 实验说明](official_eval_example/README.md)。它检查原生轨迹与 ROUGE。

路线 B：从 `google-adk` 目录运行售后质量脚本，读取仓库 `.env` 中的密钥，同时检查业务账本：

```powershell
# 这条 guard 案例会在调用模型前拦截；其他案例通常会产生费用。
..\.venv\Scripts\python.exe -m eval_example.run_quality_eval --case prompt_injection --report-dir eval_example/reports/smoke
# 从一个真实查询开始，验证模型/工具集成。
..\.venv\Scripts\python.exe -m eval_example.run_quality_eval --case order_delivered --report-dir eval_example/reports/model-smoke
# 全部 26 条，会产生多次模型调用，按实际预算决定是否执行。
..\.venv\Scripts\python.exe -m eval_example.run_quality_eval --report-dir eval_example/reports/full
```

不要将一条冒烟报告当成全量基线：

| 字段 | 冒烟 | 完整评估 |
|---|---|---|
| `selected_cases_passed` | 所选案例结果 | 所有案例结果 |
| `passed` / 退出码 | 所选案例是否通过 | 完整门禁是否通过 |
| `release_ready` | 始终 false | 所有完整门禁通过才 true |
| 无安全案例 | 安全率 null，显示未评估 | 不能通过安全门禁 |

`release_ready` 只是此教学流程的门禁结果；真实发布还需项目自己的检查。

## 8. 回归与迁移练习

有一份完整基线后，用 `--baseline <报告路径>` 比较相同数据集、相同案例集合。90%→85% 是下降 **5 个百分点**，90%→84% 下降 6 个百分点；配置 `0.05` 表示前者。冒烟不能作为基线。

现在做一个小迁移：增加“用户提供了不完整订单号 A100”的第 4 条案例。

1. 先写期望：澄清、零工具调用，不猜订单。
2. 先运行新增案例；若失败，读实际轨迹，再决定改模型逻辑还是修正不合理的期望。
3. 修复后回归原三条，确认正常查询没有退化。
4. 提交数据、实现、失败前后的报告，并解释这个案例仍没有覆盖什么。

**验收证据**：能展示一次明确的失败及其原因；能区分工具轨迹、文本相似度和业务效果；能解释离线通过不代表真实模型通过。没有失败报告，只有“我感觉改好了”，不能完成这项练习。

对于后续退款工程，把这里的内存快照改为受控测试数据库的事务前后查询，并加入并发、重试和跨进程幂等。对真实事故增加回归案例，是持续减少未知风险的方法；有限案例不能证明没有未知错误。

官方参考：[评估工作流](https://adk.dev/evaluate/) · [指标定义](https://adk.dev/evaluate/criteria/)。本章 CLI 行为同时以本地 ADK 2.6.3 实验为准。